In [8]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 80)

In [9]:
# Carga: pon NROWS=None para todo el dataset (más lento / más RAM)
# Por defecto usamos un sample más chico para que Apriori/FP-Growth terminen rápido.
NROWS = None

df = pd.read_csv("data_categorizado.csv", nrows=NROWS, low_memory=False)
df.shape

(114174, 131)

In [5]:
# Vista general rápida
display(df.head(3))
print('Columnas:', len(df.columns))
df.columns[:25]

,appID,Name,Release date,Estimated owners,Peak CCU,Price,Discount,DLC count,About the game,Supported languages,Full audio languages,Reviews,Windows,Mac,Linux,Metacritic score,Positive,Negative,Achievements,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Price_category,Release_datetime,...,tag_Indie,tag_Linear,tag_Minimalist,tag_Multiplayer,tag_Multiple Endings,tag_Mystery,tag_Open World,tag_Physics,tag_Pixel Graphics,tag_Platformer,tag_Point & Click,tag_Psychological Horror,tag_Puzzle,tag_PvE,tag_RPG,tag_Realistic,tag_Relaxing,tag_Retro,tag_Sci-fi,tag_Shooter,tag_Simulation,tag_Singleplayer,tag_Story Rich,tag_Strategy,tag_Stylized,tag_Survival,tag_Third Person,tag_Top-Down,tag_VR,tag_Visual Novel
0,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,5.24,65,0,"Springtime, April: when the cherry trees come into full bloom. The protagoni...",['English'],[],NaN,True,False,False,0,252,3,0,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,Family Sharing",Adventure,"Adventure,Visual Novel,Anime,Cute",5.00-9.99,2016-07-29,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,4.99,0,0,"Immerse yourself in the most beloved, mystical and entrancing world of Edgar...","['English', 'French', 'German', 'Russian']",[],NaN,True,True,False,0,21,3,0,0,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Object,2D,Colorful,Stylized,Logic,M...",0.01-4.99,2019-05-06,...,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0
2,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' It's been three years since I'...",['Korean'],['Korean'],NaN,True,False,False,0,0,0,19,0,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,5.00-9.99,2024-10-31,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Columnas: 131


Index(['appID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Price', 'Discount', 'DLC count', 'About the game',
       'Supported languages', 'Full audio languages', 'Reviews', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Positive', 'Negative',
       'Achievements', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks',
       'Developers', 'Publishers'],
      dtype='str')

In [10]:

# ============================================================================
# PREPARACIÓN DE DATOS PARA CLUSTERING
# ============================================================================
# Variables de éxito a utilizar:
# 1. Positive: Porcentaje de reseñas positivas
# 2. Estimated owners: Número de propietarios estimados
# 3. Peak CCU: Jugadores simultáneos máximos

print("Información de las variables de éxito:")
print(f"Columnas disponibles: {df.columns.tolist()}")
print("\n" + "="*60)

# Seleccionar las columnas relevantes
clustering_vars = ['Positive', 'Estimated owners', 'Peak CCU']
df_cluster = df[clustering_vars].copy()

print(f"\nDataset original: {df.shape}")
print(f"\nDatos seleccionados para clustering:")
print(df_cluster.head())
print(f"\nTipos de datos originales:")
print(df_cluster.dtypes)

# Limpiar 'Estimated owners': convertir rangos de texto a números (usar punto medio)
def clean_estimated_owners(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, str):
        try:
            # Si es un rango como "0 - 20000", extraer los números
            if ' - ' in val:
                parts = val.split(' - ')
                lower = float(parts[0].strip())
                upper = float(parts[1].strip())
                return (lower + upper) / 2  # Usar el punto medio
            else:
                return float(val)
        except:
            return np.nan
    return float(val)

df_cluster['Estimated owners'] = df_cluster['Estimated owners'].apply(clean_estimated_owners)

# Convertir a numérico
df_cluster['Positive'] = pd.to_numeric(df_cluster['Positive'], errors='coerce')
df_cluster['Peak CCU'] = pd.to_numeric(df_cluster['Peak CCU'], errors='coerce')

print(f"\nDatos después de limpieza de tipos:")
print(df_cluster.head())
print(f"\nEstadísticas descriptivas:")
print(df_cluster.describe())
print(f"\nValores faltantes:")
print(df_cluster.isnull().sum())


Información de las variables de éxito:
Columnas disponibles: ['appID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Price', 'Discount', 'DLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Positive', 'Negative', 'Achievements', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Price_category', 'Release_datetime', 'Release_year', 'Release_month', 'Release_season', 'total_reviews', 'review_rate', 'review_score_cat', 'sales_cat', 'ccu_cat', 'metacritic_cat', 'genre_360 Video', 'genre_Accounting', 'genre_Action', 'genre_Adventure', 'genre_Animation & Modeling', 'genre_Audio Production', 'genre_Casual', 'genre_Design & Illustration', 'genre_Documentary', 'genre_Early Access', 'genre_Education', 'genre_Episodic', 'genre_Free To Play', 'genre_Game Development', 'genre_Gore', 

In [11]:

# Limpieza: eliminar filas con valores faltantes en las variables de interés
df_cluster_clean = df_cluster.dropna()

print(f"\nDatos después de eliminar valores faltantes: {df_cluster_clean.shape}")
print(f"Porcentaje de datos retenidos: {len(df_cluster_clean) / len(df_cluster) * 100:.2f}%")
print(f"\nEstadísticas después de limpieza:")
print(df_cluster_clean.describe())

# Normalización: StandardScaler para que todas las variables tengan la misma escala
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_cluster_clean)

print(f"\nDatos normalizados (primeras 5 filas):")
print(df_scaled[:5])
print(f"\nMedia (debe ser ~0): {df_scaled.mean(axis=0)}")
print(f"Desv. Std (debe ser ~1): {df_scaled.std(axis=0)}")



Datos después de eliminar valores faltantes: (114174, 3)
Porcentaje de datos retenidos: 100.00%

Estadísticas después de limpieza:
           Positive  Estimated owners      Peak CCU
count  1.141740e+05      1.141740e+05  1.141740e+05
mean   1.122206e+03      9.205274e+04  5.862755e+01
std    2.910969e+04      1.336266e+06  3.864762e+03
min    0.000000e+00      0.000000e+00  0.000000e+00
25%    0.000000e+00      1.000000e+04  0.000000e+00
50%    7.000000e+00      1.000000e+04  0.000000e+00
75%    4.400000e+01      1.000000e+04  0.000000e+00
max    7.642084e+06      1.500000e+08  1.013936e+06

Datos normalizados (primeras 5 filas):
[[-0.02989416 -0.06140475 -0.01516984]
 [-0.0378297  -0.06140475 -0.01516984]
 [-0.03855111 -0.06140475 -0.01491109]
 [-0.03855111 -0.06140475 -0.01516984]
 [-0.03855111 -0.06140475 -0.01516984]]

Media (debe ser ~0): [9.95733159e-19 2.98719948e-18 1.49359974e-18]
Desv. Std (debe ser ~1): [1. 1. 1.]


In [16]:

# ============================================================================
# ALGORITMO 1: K-MEANS (Clustering Particional - Rápido y eficiente en memoria)
# ============================================================================
import gc

print("\n" + "="*70)
print("K-MEANS - Clustering particional optimizado para datasets grandes")
print("="*70)

from sklearn.cluster import KMeans

print(f"\nDataset: {df_scaled.shape[0]} muestras, {df_scaled.shape[1]} características")
print("Ventajas: O(n*k*d*i) complejidad, ideal para datasets grandes")

# K-Means con elbow method para encontrar k óptima
print("\n⚙ Búsqueda del número óptimo de clusters usando Elbow Method...")

inertias = []
silhouette_scores_kmeans = []
n_clusters_range = range(2, 9)

# Usar muestra consistente para Silhouette Score
np.random.seed(42)
silhouette_sample_indices = np.random.choice(len(df_scaled), size=min(10000, len(df_scaled)), replace=False)
silhouette_sample_data = df_scaled[silhouette_sample_indices]

for n_clusters in n_clusters_range:
    kmeans = KMeans(
        n_clusters=n_clusters, 
        random_state=42, 
        n_init=10,
        max_iter=300
    )
    kmeans.fit(df_scaled)
    inertias.append(kmeans.inertia_)
    
    # Calcular silhouette score en muestra
    labels_sample = kmeans.predict(silhouette_sample_data)
    if len(np.unique(labels_sample)) > 1:
        score = silhouette_score(silhouette_sample_data, labels_sample)
    else:
        score = -1.0  # Penalizar si solo hay 1 cluster
    silhouette_scores_kmeans.append(score)
    print(f"  n_clusters={n_clusters}: Inertia={kmeans.inertia_:.2f}, Silhouette (muestra)={score:.4f}")

# Usar elbow method: donde disminuye menos la inercia
optimal_n_kmeans = n_clusters_range[np.argmax(silhouette_scores_kmeans)]

print(f"\n✓ Número óptimo encontrado: {optimal_n_kmeans} clusters (máximo Silhouette)")

# Aplicar K-Means final
print(f"\n🔄 Aplicando K-Means al dataset completo con n_clusters={optimal_n_kmeans}...")
kmeans_final = KMeans(
    n_clusters=optimal_n_kmeans,
    random_state=42,
    n_init=20,
    max_iter=300
)

labels_dbscan = kmeans_final.fit_predict(df_scaled)

gc.collect()

n_clusters_dbscan = optimal_n_kmeans
n_noise_dbscan = 0  # K-Means no produce ruido

print(f"\n✓ Resultados K-Means:")
print(f"  Clusters encontrados: {n_clusters_dbscan}")
print(f"  Puntos de ruido: 0 (todos asignados)")
print(f"  Inercia final: {kmeans_final.inertia_:.2f}")
print(f"  Distribución de puntos por cluster:")

for cluster_id in range(n_clusters_dbscan):
    count = list(labels_dbscan).count(cluster_id)
    print(f"    Cluster {cluster_id}: {count} puntos ({count/len(labels_dbscan)*100:.2f}%)")

# Calcular métrica de calidad
silhouette_kmeans = silhouette_score(df_scaled, labels_dbscan)
print(f"\n✓ Métricas del K-Means (dataset completo):")
print(f"  Silhouette Score: {silhouette_kmeans:.4f}")



K-MEANS - Clustering particional optimizado para datasets grandes

Dataset: 114174 muestras, 3 características
Ventajas: O(n*k*d*i) complejidad, ideal para datasets grandes

⚙ Búsqueda del número óptimo de clusters usando Elbow Method...
  n_clusters=2: Inertia=170380.51, Silhouette (muestra)=-1.0000
  n_clusters=3: Inertia=92984.07, Silhouette (muestra)=0.9962
  n_clusters=4: Inertia=59509.42, Silhouette (muestra)=0.9962
  n_clusters=5: Inertia=43720.07, Silhouette (muestra)=0.9907
  n_clusters=6: Inertia=34286.44, Silhouette (muestra)=0.9861
  n_clusters=7: Inertia=28835.07, Silhouette (muestra)=0.9793
  n_clusters=8: Inertia=23745.83, Silhouette (muestra)=0.9793

✓ Número óptimo encontrado: 3 clusters (máximo Silhouette)

🔄 Aplicando K-Means al dataset completo con n_clusters=3...

✓ Resultados K-Means:
  Clusters encontrados: 3
  Puntos de ruido: 0 (todos asignados)
  Inercia final: 92984.07
  Distribución de puntos por cluster:
    Cluster 0: 114127 puntos (99.96%)
    Cluster 1:

In [17]:

# ============================================================================
# ALGORITMO 2: GMM (Gaussian Mixture Model)
# ============================================================================
from sklearn.mixture import GaussianMixture
from scipy.cluster.hierarchy import dendrogram, linkage

print("\n" + "="*70)
print("GMM - Modelo de Mezcla Gaussiana")
print("="*70)

# Buscar el número óptimo de componentes usando BIC y AIC
n_components_range = range(2, 11)
bic_scores = []
aic_scores = []

for n_components in n_components_range:
    gmm = GaussianMixture(n_components=n_components, random_state=42, n_init=10)
    gmm.fit(df_scaled)
    bic_scores.append(gmm.bic(df_scaled))
    aic_scores.append(gmm.aic(df_scaled))

optimal_n_gmm = n_components_range[np.argmin(bic_scores)]

print(f"\nBúsqueda de número óptimo de componentes:")
print(f"  BIC (criterio Bayesiano de información):")
for n, bic in zip(n_components_range, bic_scores):
    marker = " <-- Óptimo" if n == optimal_n_gmm else ""
    print(f"    n_components={n}: BIC={bic:.2f}{marker}")

# Aplicar GMM con número óptimo de componentes
gmm = GaussianMixture(n_components=optimal_n_gmm, random_state=42, n_init=10, covariance_type='full')
labels_gmm = gmm.fit_predict(df_scaled)
probabilities_gmm = gmm.predict_proba(df_scaled)

print(f"\nParametros utilizados:")
print(f"  n_components: {optimal_n_gmm}")
print(f"  covariance_type: full")
print(f"\nResultados GMM:")
print(f"  Log-Likelihood: {gmm.score(df_scaled):.4f}")
print(f"  BIC: {gmm.bic(df_scaled):.4f}")
print(f"  Distribución de puntos por componente:")
for component_id in range(optimal_n_gmm):
    count = list(labels_gmm).count(component_id)
    print(f"    Componente {component_id}: {count} puntos ({count/len(labels_gmm)*100:.2f}%)")

print(f"\n  Probabilidades promedio de asignación: {probabilities_gmm.max(axis=1).mean():.4f}")



GMM - Modelo de Mezcla Gaussiana

Búsqueda de número óptimo de componentes:
  BIC (criterio Bayesiano de información):
    n_components=2: BIC=-2557449.81
    n_components=3: BIC=-2589528.65
    n_components=4: BIC=-2599844.58
    n_components=5: BIC=-2912654.67
    n_components=6: BIC=-3049785.60
    n_components=7: BIC=-3072747.89
    n_components=8: BIC=-3090260.03 <-- Óptimo
    n_components=9: BIC=-3073973.57
    n_components=10: BIC=-3075716.22

Parametros utilizados:
  n_components: 8
  covariance_type: full

Resultados GMM:
  Log-Likelihood: 13.5371
  BIC: -3090260.0253
  Distribución de puntos por componente:
    Componente 0: 85172 puntos (74.60%)
    Componente 1: 1 puntos (0.00%)
    Componente 2: 1219 puntos (1.07%)
    Componente 3: 2 puntos (0.00%)
    Componente 4: 7168 puntos (6.28%)
    Componente 5: 16907 puntos (14.81%)
    Componente 6: 3401 puntos (2.98%)
    Componente 7: 304 puntos (0.27%)

  Probabilidades promedio de asignación: 0.9846


In [ ]:

# ============================================================================
# COMPARACIÓN Y ANÁLISIS DE LOS DOS ALGORITMOS
# ============================================================================
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

print("\n" + "="*70)
print("COMPARACIÓN DE ALGORITMOS DE CLUSTERING")
print("="*70)

# Calcular métricas de evaluación para cada algoritmo
algorithms = {
    'K-Means': labels_dbscan,
    'GMM': labels_gmm
}

def compute_metrics(labels, data, algorithm_name):
    if len(set(labels)) > 1:
        silhouette = silhouette_score(data, labels)
        calinski = calinski_harabasz_score(data, labels)
        davies_bouldin = davies_bouldin_score(data, labels)
        return silhouette, calinski, davies_bouldin
    else:
        return np.nan, np.nan, np.nan

print("\nMétricas de evaluación de clustering:")
print(f"{'Algoritmo':<15} {'Silhouette':<15} {'Calinski-Harabasz':<20} {'Davies-Bouldin':<18}")
print("-" * 70)

results = {}
for algo_name, labels in algorithms.items():
    silhouette, calinski, davies_bouldin = compute_metrics(labels, df_scaled, algo_name)
    results[algo_name] = {
        'silhouette': silhouette,
        'calinski': calinski,
        'davies_bouldin': davies_bouldin,
        'labels': labels
    }
    print(f"{algo_name:<15} {silhouette:<15.4f} {calinski:<20.2f} {davies_bouldin:<18.4f}")

print("\nInterpretación de métricas:")
print("  - Silhouette Score: Rango [-1, 1]. Más alto = mejor. (>0.5 es bueno)")
print("  - Calinski-Harabasz: Más alto = mejor. (>10 es generalmente bueno)")
print("  - Davies-Bouldin: Más bajo = mejor. (<1 es muy bueno, <2 es aceptable)")

# Resumen de características de cada algoritmo
print("\n" + "="*70)
print("RESUMEN DE CARACTERÍSTICAS DE CADA ALGORITMO")
print("="*70)

summary_text = """
1. K-MEANS (Clustering Particional):
   ✓ Ventajas:
     - Muy rápido: O(n*k*d*i)
     - Escalable para datasets grandes
     - Todos los puntos asignados a clusters
     - Simple de implementar y entender
   ✗ Desventajas:
     - Requiere especificar k a priori
     - Sensible a inicialización
     - Asume clusters esféricos
     - Puede no capturar estructuras complejas

2. GMM (Gaussian Mixture Model):
   ✓ Ventajas:
     - Modelo probabilístico: proporciona probabilidades de pertenencia
     - Flexibilidad en forma de clusters
     - Usa criterios estadísticos (BIC, AIC) para seleccionar n_components
     - Basado en principios estadísticos sólidos
   ✗ Desventajas:
     - Asume que clusters son gaussianos
     - Computacionalmente más pesado
     - Requiere más datos para convergencia
     - Convergencia puede ser lenta en datasets grandes
"""
print(summary_text)

print("\n" + "="*70)
print("RECOMENDACIÓN")
print("="*70)
best_algo = max(results.items(), key=lambda x: x[1]['silhouette'])
print(f"\nAlgoritmo con mejor Silhouette Score: {best_algo[0]} ({best_algo[1]['silhouette']:.4f})")
print(f"Este algoritmo proporciona la mejor separación y cohesión de clusters para este dataset.")



COMPARACIÓN DE ALGORITMOS DE CLUSTERING

Métricas de evaluación de clustering:
Algoritmo       Silhouette      Calinski-Harabasz    Davies-Bouldin    
----------------------------------------------------------------------


In [ ]:

# ============================================================================
# VISUALIZACIONES 3D DE LOS RESULTADOS
# ============================================================================
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

print("\n" + "="*70)
print("VISUALIZACIONES 3D DE CLUSTERING")
print("="*70)

# Crear figura con 2 subplots para los dos algoritmos
fig = plt.figure(figsize=(14, 5))

# Variables para plotear: Positive, Estimated owners, Peak CCU
var_names = clustering_vars

# Plot 1: K-Means
ax1 = fig.add_subplot(121, projection='3d')
scatter1 = ax1.scatter(df_scaled[:, 0], 
                       df_scaled[:, 1], 
                       df_scaled[:, 2],
                       c=labels_dbscan, 
                       cmap='viridis', 
                       s=30, 
                       alpha=0.6)
ax1.set_xlabel(var_names[0])
ax1.set_ylabel(var_names[1])
ax1.set_zlabel(var_names[2])
ax1.set_title(f'K-Means\n(Clusters: {n_clusters_dbscan})')
plt.colorbar(scatter1, ax=ax1, label='Cluster')

# Plot 2: GMM
ax2 = fig.add_subplot(122, projection='3d')
scatter2 = ax2.scatter(df_scaled[:, 0], 
                       df_scaled[:, 1], 
                       df_scaled[:, 2],
                       c=labels_gmm, 
                       cmap='viridis', 
                       s=30, 
                       alpha=0.6)
ax2.set_xlabel(var_names[0])
ax2.set_ylabel(var_names[1])
ax2.set_zlabel(var_names[2])
ax2.set_title(f'GMM\n(Componentes: {optimal_n_gmm})')
plt.colorbar(scatter2, ax=ax2, label='Componente')

plt.tight_layout()
plt.show()

print("Visualización 3D completada")


In [ ]:

# ============================================================================
# ANÁLISIS PROFUNDO: CARACTERÍSTICAS DE CADA CLUSTER
# ============================================================================
print("\n" + "="*70)
print("ANÁLISIS DETALLADO DE CLUSTERS - INTERPRETACIÓN EMPRESARIAL")
print("="*70)

# Analizar GMM en detalle (probablemente el mejor para este caso)
print(f"\n### ANÁLISIS DETALLADO: GMM ###\n")

# Agregar los labels al dataframe original para análisis
df_cluster_clean_copy = df_cluster_clean.copy()
df_cluster_clean_copy['GMM_Cluster'] = labels_gmm

print("Estadísticas por componente GMM:")
print("="*70)
for component_id in range(optimal_n_gmm):
    cluster_data = df_cluster_clean_copy[df_cluster_clean_copy['GMM_Cluster'] == component_id]
    
    print(f"\nCOMPONENTE {component_id}:")
    print(f"  Tamaño: {len(cluster_data)} videojuegos ({len(cluster_data)/len(df_cluster_clean_copy)*100:.2f}%)")
    print(f"\n  Estadísticas de las variables de éxito:")
    print(f"    Positive (% reseñas positivas):")
    print(f"      - Media: {cluster_data['Positive'].mean():.2f}%")
    print(f"      - Mediana: {cluster_data['Positive'].median():.2f}%")
    print(f"      - Rango: [{cluster_data['Positive'].min():.2f}, {cluster_data['Positive'].max():.2f}]")
    
    print(f"    Estimated owners (propietarios estimados):")
    print(f"      - Media: {cluster_data['Estimated owners'].mean():.0f}")
    print(f"      - Mediana: {cluster_data['Estimated owners'].median():.0f}")
    print(f"      - Rango: [{cluster_data['Estimated owners'].min():.0f}, {cluster_data['Estimated owners'].max():.0f}]")
    
    print(f"    Peak CCU (jugadores simultáneos máximos):")
    print(f"      - Media: {cluster_data['Peak CCU'].mean():.0f}")
    print(f"      - Mediana: {cluster_data['Peak CCU'].median():.0f}")
    print(f"      - Rango: [{cluster_data['Peak CCU'].min():.0f}, {cluster_data['Peak CCU'].max():.0f}]")
    
    # Clasificar el cluster
    avg_positive = cluster_data['Positive'].mean()
    avg_owners = cluster_data['Estimated owners'].mean()
    avg_peak = cluster_data['Peak CCU'].mean()
    
    print(f"\n  CLASIFICACIÓN (relativa a la media general):")
    print(f"    Éxito de reseñas: {'🟢 ALTO' if avg_positive > df_cluster_clean['Positive'].mean() else '🔴 BAJO'}")
    print(f"    Éxito de ventas: {'🟢 ALTO' if avg_owners > df_cluster_clean['Estimated owners'].mean() else '🔴 BAJO'}")
    print(f"    Éxito de jugadores simultáneos: {'🟢 ALTO' if avg_peak > df_cluster_clean['Peak CCU'].mean() else '🔴 BAJO'}")
    
    # Asignar un nombre interpretativo
    if avg_positive > df_cluster_clean['Positive'].mean() and avg_owners > df_cluster_clean['Estimated owners'].mean():
        cluster_type = "ÉXITO ALTO (Críticas y Ventas)"
    elif avg_peak > df_cluster_clean['Peak CCU'].mean():
        cluster_type = "MULTIPLAYER EXITOSO"
    elif avg_owners > df_cluster_clean['Estimated owners'].mean():
        cluster_type = "VENTAS ALTAS (sin multiplejador destacado)"
    elif avg_positive > df_cluster_clean['Positive'].mean():
        cluster_type = "CRÍTICAS POSITIVAS (nicho)"
    else:
        cluster_type = "BAJO PERFIL / NICHO"
    
    print(f"\n  ➜ PERFIL: {cluster_type}")
    print("-" * 70)
